<a href="https://colab.research.google.com/github/AhmadSyahreza/Katanya-kami-lomba/blob/main/Allnew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LOAD DATA

In [43]:
# CELL 2 — Config & imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pd.set_option("display.max_columns", 50)
np.random.seed(42)
os.makedirs("plots", exist_ok=True)

@dataclass
class Config:
    date_col: str = "date_show"
    target_col: str = "total_ticket"
    id_col: str = "id"
    group_cols: List[str] = field(default_factory=lambda: ["cinema_ids", "movie_title"])
    city_col: str = "city_name"
    val_days: int = 30
    lags: List[int] = field(default_factory=lambda: [1, 2, 3, 7, 14, 28])
    roll_windows: List[int] = field(default_factory=lambda: [7, 14, 28])
    seasonality: int = 7      # harian, mingguan
    horizon: int = 1

cfg = Config()
cfg

Config(date_col='date_show', target_col='total_ticket', id_col='id', group_cols=['cinema_ids', 'movie_title'], city_col='city_name', val_days=30, lags=[1, 2, 3, 7, 14, 28], roll_windows=[7, 14, 28], seasonality=7, horizon=1)

In [44]:
# # CELL 3a: Mount Drive & load
# from google.colab import drive
# drive.mount('/content/drive')

# import pandas as pd
# import numpy as np

# BASE = '/content/drive/MyDrive/Lomba Data'

# train1         = pd.read_csv(f'{BASE}/train.csv')
# test1          = pd.read_csv(f'{BASE}/test.csv')
# holidays       = pd.read_csv(f'{BASE}/holidays.csv')
# movies         = pd.read_csv(f'{BASE}/movies.csv')
# ticket_prices  = pd.read_csv(f'{BASE}/ticket_prices.csv')
# test_history   = pd.read_csv(f'{BASE}/test_history.csv')
# sample_submission = pd.read_csv(f'{BASE}/sample_submission.csv')

# files = {
#     'train1': train1,
#     'test1': test1,
#     'holidays': holidays,
#     'movies': movies,
#     'ticket_prices': ticket_prices,
#     'test_history': test_history,
#     'sample_submission': sample_submission,
# }
# print("Loaded.")
# for k, v in files.items():
#     print(f"  {k:20s} shape={v.shape}")

In [45]:
# CELL 3 — Load data
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Lomba Data'

train1            = pd.read_csv(f'{BASE}/train.csv')
test1             = pd.read_csv(f'{BASE}/test.csv')
holidays          = pd.read_csv(f'{BASE}/holidays.csv')
movies            = pd.read_csv(f'{BASE}/movies.csv')
ticket_prices     = pd.read_csv(f'{BASE}/ticket_prices.csv')
test_history      = pd.read_csv(f'{BASE}/test_history.csv')
sample_submission = pd.read_csv(f'{BASE}/sample_submission.csv')

for k, v in [('train1', train1), ('test1', test1), ('holidays', holidays),
             ('movies', movies), ('ticket_prices', ticket_prices),
             ('test_history', test_history), ('sample_submission', sample_submission)]:
    print(f"{k:20s} shape={v.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train1               shape=(138959, 7)
test1                shape=(72611, 5)
holidays             shape=(366, 4)
movies               shape=(397, 7)
ticket_prices        shape=(207, 3)
test_history         shape=(32323, 7)
sample_submission    shape=(72611, 2)


In [46]:
# CELL 3-NEW — Audit data asli
tr = train1.copy()
te = test1.copy()
tr[cfg.date_col] = pd.to_datetime(tr[cfg.date_col])
te[cfg.date_col] = pd.to_datetime(te[cfg.date_col])

print("="*60)
print("RENTANG TANGGAL")
print("="*60)
print(f"Train: {tr[cfg.date_col].min()} -> {tr[cfg.date_col].max()}  n={len(tr)}")
print(f"Test : {te[cfg.date_col].min()} -> {te[cfg.date_col].max()}  n={len(te)}")

print("\n" + "="*60)
print("GRUP & KARDINALITAS")
print("="*60)
print(f"Cinema unik       : {tr['cinema_ids'].nunique()}")
print(f"Film unik (train) : {tr['movie_title'].nunique()}")
print(f"Film unik (test)  : {te['movie_title'].nunique()}")
print(f"Kombinasi cinema-film di train: {tr.groupby(cfg.group_cols).ngroups}")

train_movies = set(tr['movie_title'].unique())
test_movies  = set(te['movie_title'].unique())
new_movies   = test_movies - train_movies
print(f"\nFilm test TIDAK ada di train : {len(new_movies)}")
print(f"Film test ada di train       : {len(test_movies & train_movies)}")

new_rows = te[~te['movie_title'].isin(train_movies)]
print(f"Baris test dengan film BARU  : {len(new_rows)} / {len(te)} "
      f"({100*len(new_rows)/len(te):.1f}%)")

print("\n" + "="*60)
print("CEK FILE PENDUKUNG")
print("="*60)
print("holidays kolom     :", list(holidays.columns))
print("movies kolom       :", list(movies.columns))
print("ticket_prices kolom:", list(ticket_prices.columns))
print("test_history kolom :", list(test_history.columns))
print("test_history shape :", test_history.shape)
print("sample_submission  :", list(sample_submission.columns))

RENTANG TANGGAL
Train: 2025-04-01 00:00:00 -> 2025-09-30 00:00:00  n=138959
Test : 2025-10-04 00:00:00 -> 2026-03-27 00:00:00  n=72611

GRUP & KARDINALITAS
Cinema unik       : 117
Film unik (train) : 237
Film unik (test)  : 160
Kombinasi cinema-film di train: 13041

Film test TIDAK ada di train : 150
Film test ada di train       : 10
Baris test dengan film BARU  : 67095 / 72611 (92.4%)

CEK FILE PENDUKUNG
holidays kolom     : ['date', 'day_tipe', 'holiday_tipe', 'holiday_name']
movies kolom       : ['original_title', 'age_rating', 'genre', 'producer', 'director', 'writer', 'casts']
ticket_prices kolom: ['city_name', 'ceil', 'price_day']
test_history kolom : ['date_show', 'cinema_ids', 'city_name', 'movie_title', 'total_ticket', 'occupation_rate', 'total_show']
test_history shape : (32323, 7)
sample_submission  : ['id', 'total_ticket']


In [47]:
# CELL 3b: Audit tiap file
def audit_df(name, df, n=3):
    print("=" * 70)
    print(f"FILE: {name}   shape={df.shape}")
    print("=" * 70)
    print(f"Kolom : {list(df.columns)}")
    print(f"Dtypes:\n{df.dtypes}")
    print(f"Missing:\n{df.isna().sum()}")
    print(f"Head({n}):")
    print(df.head(n))
    print()

for k, v in files.items():
    audit_df(k, v)

FILE: train1   shape=(138959, 7)
Kolom : ['date_show', 'cinema_ids', 'city_name', 'movie_title', 'total_ticket', 'occupation_rate', 'total_show']
Dtypes:
date_show           object
cinema_ids          object
city_name           object
movie_title         object
total_ticket         int64
occupation_rate    float64
total_show           int64
dtype: object
Missing:
date_show          0
cinema_ids         0
city_name          0
movie_title        0
total_ticket       0
occupation_rate    0
total_show         0
dtype: int64
Head(3):
    date_show                        cinema_ids city_name    movie_title  \
0  2025-04-01  015555fbcd8faef6b55a854605df0a87  SURABAYA  A WORKING MAN   
1  2025-04-01  015555fbcd8faef6b55a854605df0a87  SURABAYA          JUMBO   
2  2025-04-01  015555fbcd8faef6b55a854605df0a87  SURABAYA         KOMANG   

   total_ticket  occupation_rate  total_show  
0           487            31.07          18  
1           649            23.69          19  
2           389    

In [48]:
# CELL 3c: Deteksi kolom tanggal & target
def guess_date_cols(df):
    cols = []
    for c in df.columns:
        if 'date' in c.lower() or 'time' in c.lower() or 'dt' in c.lower():
            cols.append(c)
        else:
            # coba parse kalau kolomnya string
            if df[c].dtype == object:
                sample = df[c].dropna().head(5).astype(str).tolist()
                if sample and all(any(ch.isdigit() for ch in s) and ('-' in s or '/' in s) for s in sample):
                    cols.append(c)
    return cols

for k, v in files.items():
    print(f"{k:20s} kandidat kolom tanggal: {guess_date_cols(v)}")

train1               kandidat kolom tanggal: ['date_show']
test1                kandidat kolom tanggal: ['date_show']
holidays             kandidat kolom tanggal: ['date']
movies               kandidat kolom tanggal: []
ticket_prices        kandidat kolom tanggal: []
test_history         kandidat kolom tanggal: ['date_show']
sample_submission    kandidat kolom tanggal: []


In [49]:
# CELL 3d: Isi manual sesuai hasil audit
# --- dari output CELL 3b, isi yang benar ---

# Kolom identitas di train/test
ID_COL     = 'id'          # ganti kalau beda
DATE_COL   = 'date'        # ganti kalau beda
TARGET_COL = 'y'           # ganti kalau beda (kolom yang mau diprediksi)

# File test: apakah test1 punya kolom target?
# - Kalau TIDAK ada target -> ini test set asli (perlu submit)
# - Kalau ADA target -> ini mungkin public/private split
print("Kolom target ada di train?", TARGET_COL in train1.columns)
print("Kolom target ada di test? ", TARGET_COL in test1.columns)

# Apakah test_history = test dengan histori?
print("test1 shape       :", test1.shape)
print("test_history shape:", test_history.shape)
print("test_history kolom:", list(test_history.columns))

Kolom target ada di train? False
Kolom target ada di test?  False
test1 shape       : (72611, 5)
test_history shape: (32323, 7)
test_history kolom: ['date_show', 'cinema_ids', 'city_name', 'movie_title', 'total_ticket', 'occupation_rate', 'total_show']


In [50]:
# CELL 3e: Cek nilai unik
for k, v in files.items():
    print("=" * 70)
    print(f"{k} (shape={v.shape})")
    print("=" * 70)
    for c in v.columns:
        n_unique = v[c].nunique()
        if n_unique <= 20:
            print(f"  {c:20s} n_unique={n_unique:5d}  values={v[c].dropna().unique()[:20]}")
        else:
            print(f"  {c:20s} n_unique={n_unique:5d}  contoh={v[c].dropna().head(3).tolist()}")
    print()

train1 (shape=(138959, 7))
  date_show            n_unique=  182  contoh=['2025-04-01', '2025-04-01', '2025-04-01']
  cinema_ids           n_unique=  117  contoh=['015555fbcd8faef6b55a854605df0a87', '015555fbcd8faef6b55a854605df0a87', '015555fbcd8faef6b55a854605df0a87']
  city_name            n_unique=   67  contoh=['SURABAYA', 'SURABAYA', 'SURABAYA']
  movie_title          n_unique=  237  contoh=['A WORKING MAN', 'JUMBO', 'KOMANG']
  total_ticket         n_unique= 4050  contoh=[487, 649, 389]
  occupation_rate      n_unique= 7866  contoh=[31.07, 23.69, 9.55]
  total_show           n_unique=  196  contoh=[18, 19, 22]

test1 (shape=(72611, 5))
  id                   n_unique=72611  contoh=[1, 2, 3]
  movie_title          n_unique=  160  contoh=['13 DAYS, 13 NIGHTS', '13 DAYS, 13 NIGHTS', '13 DAYS, 13 NIGHTS']
  cinema_ids           n_unique=  121  contoh=['015555fbcd8faef6b55a854605df0a87', '015555fbcd8faef6b55a854605df0a87', '015555fbcd8faef6b55a854605df0a87']
  city_name            n_

In [51]:
# CELL 3f: Rentang tanggal tiap file yang punya kolom tanggal
for k, v in files.items():
    date_cols = guess_date_cols(v)
    for c in date_cols:
        try:
            s = pd.to_datetime(v[c], errors='coerce')
            if s.notna().any():
                print(f"{k}.{c}: {s.min()} -> {s.max()}  (NaN: {s.isna().sum()})")
        except Exception as e:
            print(f"{k}.{c}: gagal parse ({e})")

train1.date_show: 2025-04-01 00:00:00 -> 2025-09-30 00:00:00  (NaN: 0)
test1.date_show: 2025-10-04 00:00:00 -> 2026-03-27 00:00:00  (NaN: 0)
holidays.date: 2025-04-01 00:00:00 -> 2026-04-01 00:00:00  (NaN: 0)
test_history.date_show: 2025-10-01 00:00:00 -> 2026-03-20 00:00:00  (NaN: 0)


In [52]:
# CELL 3g: Overlap train/test? Data waktu tidak boleh overlap
if DATE_COL in train1.columns and DATE_COL in test1.columns:
    tr_dates = pd.to_datetime(train1[DATE_COL])
    te_dates = pd.to_datetime(test1[DATE_COL])
    print("Train:", tr_dates.min(), "->", tr_dates.max())
    print("Test :", te_dates.min(), "->", te_dates.max())
    overlap = set(tr_dates).intersection(set(te_dates))
    print(f"Overlap train/test: {len(overlap)} tanggal")
    if overlap:
        print("  [!!] Ada overlap -> cek apakah test bocor ke train")
    else:
        print("  [ok] Tidak ada overlap -> test murni masa depan")

In [53]:
# CELL 3: Load data
# >>> GANTI fungsi ini dengan pembacaan file kamu, contoh:
# df_raw = pd.read_csv("data.csv", parse_dates=[cfg.date_col])

def load_demo():
    rng = pd.date_range("2018-01-01", "2024-12-01", freq=cfg.freq)
    n = len(rng)
    t = np.arange(n)
    y = (100
         + 0.5 * t
         + 10 * np.sin(2 * np.pi * t / 12)
         + np.random.default_rng(42).normal(0, 3, n))
    df = pd.DataFrame({cfg.date_col: rng, cfg.target_col: y})
    df.loc[5:6, cfg.target_col] = np.nan    # simulasi missing
    df.loc[20, cfg.target_col] = 9999       # simulasi outlier
    return df

df_raw = load_demo()
df_raw.head()

AttributeError: 'Config' object has no attribute 'freq'

In [55]:
# CELL 4: Audit data mentah
def audit_basic(df, cfg):
    print("=" * 60)
    print("AUDIT DATA MENTAH")
    print("=" * 60)
    print(f"Shape            : {df.shape}")
    print(f"Kolom            : {list(df.columns)}")
    print(f"Tipe data        :\n{df.dtypes}")
    print(f"Missing per kolom:\n{df.isna().sum()}")
    print(f"Duplikat tanggal : {df[cfg.date_col].duplicated().sum()}")
    print(f"Range tanggal    : {df[cfg.date_col].min()} -> {df[cfg.date_col].max()}")
    print(f"Target describe  :\n{df[cfg.target_col].describe()}")
    print("=" * 60)

audit_basic(df_raw, cfg)

AUDIT DATA MENTAH
Shape            : (84, 2)
Kolom            : ['date', 'y']
Tipe data        :
date    datetime64[ns]
y              float64
dtype: object
Missing per kolom:
date    0
y       2
dtype: int64


KeyError: 'date_show'

In [57]:
# CELL 5 (FIX) — Cleaning
def clean(df, cfg):
    df = df.copy()
    df[cfg.date_col] = pd.to_datetime(df[cfg.date_col])
    df = df.sort_values(cfg.date_col).reset_index(drop=True)

    dup = df[cfg.date_col].duplicated().sum()
    if dup:
        print(f"[clean] hapus {dup} baris duplikat tanggal")
        df = df.drop_duplicates(subset=cfg.date_col, keep="last")

    # Reindex ke frekuensi penuh — index jadi DatetimeIndex
    df = df.set_index(cfg.date_col).asfreq(cfg.freq)

    # Outlier: rolling median (hanya ke belakang)
    y = df[cfg.target_col]
    roll_med = y.rolling(window=cfg.seasonality, min_periods=4).median()
    roll_mad = (y - roll_med).abs().rolling(window=cfg.seasonality, min_periods=4).median()
    mask = (y - roll_med).abs() > 6 * roll_mad.replace(0, np.nan)
    print(f"[clean] outlier diganti rolling-median: {int(mask.sum())}")
    df[cfg.target_col] = y.mask(mask, roll_med)

    # Interpolasi waktu — index sudah DatetimeIndex, aman
    before = int(df[cfg.target_col].isna().sum())
    df[cfg.target_col] = df[cfg.target_col].interpolate(
        method="time", limit_direction="forward"
    )
    after = int(df[cfg.target_col].isna().sum())
    print(f"[clean] NaN target: {before} -> {after}")

    # Balikin tanggal jadi kolom
    df = df.reset_index()
    df = df.dropna(subset=[cfg.target_col]).reset_index(drop=True)
    return df

df = clean(df_raw, cfg)
print(df.dtypes)
print(df.head())

KeyError: 'date_show'

In [ ]:
# # CELL 5: Cleaning
# def clean(df, cfg):
#     df = df.copy()
#     df[cfg.date_col] = pd.to_datetime(df[cfg.date_col])
#     df = df.sort_values(cfg.date_col).reset_index(drop=True)

#     dup = df[cfg.date_col].duplicated().sum()
#     if dup:
#         print(f"[clean] hapus {dup} baris duplikat tanggal")
#         df = df.drop_duplicates(subset=cfg.date_col, keep="last")

#     # Reindex ke frekuensi penuh (tanggal hilang jadi NaN)
#     df = df.set_index(cfg.date_col).asfreq(cfg.freq).reset_index()

#     # Outlier: pakai ROLLING median (hanya ke belakang), bukan median global
#     y = df[cfg.target_col]
#     roll_med = y.rolling(window=cfg.seasonality, min_periods=4).median()
#     roll_mad = (y - roll_med).abs().rolling(window=cfg.seasonality, min_periods=4).median()
#     mask = (y - roll_med).abs() > 6 * roll_mad.replace(0, np.nan)
#     print(f"[clean] outlier diganti rolling-median: {int(mask.sum())}")
#     df[cfg.target_col] = y.mask(mask, roll_med)

#     # Missing: interpolate HANYA ke arah depan (bukan bfill!)
#     before = int(df[cfg.target_col].isna().sum())
#     df[cfg.target_col] = df[cfg.target_col].interpolate(
#         method="time", limit_direction="forward"
#     )
#     after = int(df[cfg.target_col].isna().sum())
#     print(f"[clean] NaN target: {before} -> {after}")

#     df = df.dropna(subset=[cfg.target_col]).reset_index(drop=True)
#     return df

# df = clean(df_raw, cfg)
# df.head()

In [ ]:
# CELL 6: EDA
def run_eda(df, cfg):
    fig, ax = plt.subplots(3, 1, figsize=(12, 10))

    ax[0].plot(df[cfg.date_col], df[cfg.target_col], marker="o", lw=1)
    ax[0].set_title("Time Series Target")
    ax[0].grid(alpha=0.3)

    s = df.set_index(cfg.date_col)[cfg.target_col].asfreq(cfg.freq)
    try:
        dec = seasonal_decompose(s, model="additive", period=cfg.seasonality)
        ax[1].plot(dec.trend, label="trend")
        ax[1].plot(dec.seasonal, label="seasonal")
        ax[1].legend(); ax[1].set_title("Decomposition")
    except Exception as e:
        ax[1].text(0.5, 0.5, f"decompose gagal: {e}", ha="center")

    plot_acf(s.dropna(), ax=ax[2], lags=min(24, len(s)//2 - 1))
    ax[2].set_title("ACF")

    plt.tight_layout()
    plt.savefig("plots/eda.png", dpi=110)
    plt.show()

    tmp = df.copy()
    tmp["month"] = pd.to_datetime(tmp[cfg.date_col]).dt.month
    print("[eda] rata-rata per bulan:")
    print(tmp.groupby("month")[cfg.target_col].mean().round(2))

run_eda(df, cfg)

In [ ]:
# CELL 7: Fitur & target
def make_features(df, cfg):
    df = df.copy()
    y = df[cfg.target_col]

    for lag in cfg.lags:
        if lag < cfg.horizon:
            raise ValueError(f"Lag {lag} < horizon {cfg.horizon} -> LEAKAGE")
        df[f"lag_{lag}"] = y.shift(lag)

    # Rolling: shift(horizon) dulu, baru rolling
    for w in cfg.roll_windows:
        df[f"rollmean_{w}"] = y.shift(cfg.horizon).rolling(w).mean()
        df[f"rollstd_{w}"]  = y.shift(cfg.horizon).rolling(w).std()

    # Momentum
    df["diff_1"]       = y.shift(cfg.horizon) - y.shift(cfg.horizon + 1)
    df["pct_change_1"] = y.shift(cfg.horizon).pct_change()

    # Kalender (deterministik, aman)
    d = pd.to_datetime(df[cfg.date_col])
    df["month"]   = d.dt.month
    df["quarter"] = d.dt.quarter
    df["year"]    = d.dt.year

    # Target: y(t+h)
    df["y_target"] = y.shift(-cfg.horizon)

    df = df.dropna().reset_index(drop=True)
    return df

df_feat = make_features(df, cfg)
print("Shape fitur:", df_feat.shape)
df_feat.head()

In [ ]:
# CELL 8: Audit leakage fitur
def audit_features(df_feat, cfg, max_corr=0.95):
    print("=" * 60)
    print("AUDIT LEAKAGE FITUR")
    print("=" * 60)

    y = df_feat["y_target"]
    drop = [cfg.target_col, "y_target", cfg.date_col]
    X = df_feat.drop(columns=drop, errors="ignore")

    for col in X.columns:
        if X[col].equals(y):
            print(f"  [!!] '{col}' identik dengan target -> LEAKAGE LANGSUNG")

    corr_y = X.corrwith(y).abs().sort_values(ascending=False)
    print("\n  Korelasi fitur vs target (top 10):")
    for col, v in corr_y.head(10).items():
        flag = "  <-- CURIGA (>0.95)" if v > max_corr else ""
        print(f"    {col:18s}: {v:.4f}{flag}")

    y_future = df_feat[cfg.target_col].shift(-1)
    corr_f = X.corrwith(y_future).abs().sort_values(ascending=False)
    print("\n  Fitur paling berkorelasi dengan y(t+1):")
    for col, v in corr_f.head(5).items():
        flag = "  <-- CURIGA: tahu masa depan?" if v > 0.99 else ""
        print(f"    {col:18s}: {v:.4f}{flag}")

    print("\n  Cek lag vs horizon:")
    for col in X.columns:
        if col.startswith("lag_"):
            lag_val = int(col.split("_")[1])
            ok = "ok " if lag_val >= cfg.horizon else "[!!] LEAKAGE"
            print(f"    {ok} '{col}' lag={lag_val} horizon={cfg.horizon}")
    print("=" * 60)

audit_features(df_feat, cfg)

In [ ]:
# CELL 9: Split temporal (train / val / test)
def temporal_split(df, cfg):
    df = df.copy()
    df[cfg.date_col] = pd.to_datetime(df[cfg.date_col])
    train = df[df[cfg.date_col] <= cfg.train_end].reset_index(drop=True)
    val   = df[(df[cfg.date_col] > cfg.train_end) & (df[cfg.date_col] <= cfg.val_end)].reset_index(drop=True)
    test  = df[df[cfg.date_col] > cfg.val_end].reset_index(drop=True)
    print(f"[split] train={len(train)}  val={len(val)}  test={len(test)}")
    return train, val, test

train, val, test = temporal_split(df_feat, cfg)

X_cols = [c for c in train.columns
          if c not in [cfg.date_col, cfg.target_col, "y_target"]]

X_train, y_train = train[X_cols], train["y_target"]
X_val,   y_val   = val[X_cols],   val["y_target"]
X_test,  y_test  = test[X_cols],  test["y_target"]

# y_train_raw dipakai untuk denominator MASE
y_train_raw = train[cfg.target_col].values
y_val_raw   = val[cfg.target_col].values
y_test_raw  = test[cfg.target_col].values

X_cols

In [ ]:
# CELL 10: Metrik
def mase(y_true, y_pred, y_train, m=12):
    y_true  = np.asarray(y_true, float)
    y_pred  = np.asarray(y_pred, float)
    y_train = np.asarray(y_train, float)

    if m <= 1:
        naive_err = np.abs(np.diff(y_train))
    else:
        naive_err = np.abs(y_train[m:] - y_train[:-m])

    scale = np.mean(naive_err)
    if not np.isfinite(scale) or scale == 0:
        raise ValueError("MASE scale 0/NaN. Cek data & seasonality.")

    return np.mean(np.abs(y_true - y_pred)) / scale

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom == 0, 1e-9, denom)
    return 100 * np.mean(np.abs(y_true - y_pred) / denom)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def report(y_true, y_pred, y_train, m, label=""):
    print(f"[{label:20s}] "
          f"MAE={mean_absolute_error(y_true, y_pred):7.3f}  "
          f"RMSE={rmse(y_true, y_pred):7.3f}  "
          f"MASE={mase(y_true, y_pred, y_train, m):6.4f}  "
          f"sMAPE={smape(y_true, y_pred):6.2f}%")

In [ ]:
# CELL 11: Baseline walk-forward
def evaluate_baselines(y_train_raw, y_val_raw, cfg):
    print("=" * 60)
    print("BASELINE (hanya pakai data masa lalu)")
    print("=" * 60)

    hist = list(y_train_raw)
    p_naive, p_snaive, p_drift = [], [], []

    for y_true in y_val_raw:
        p_naive.append(hist[-1])
        p_snaive.append(hist[-cfg.seasonality])
        p_drift.append(hist[-1] + cfg.horizon*(hist[-1]-hist[0])/(len(hist)-1))
        hist.append(y_true)

    for name, preds in [("naive", p_naive),
                        ("seasonal_naive", p_snaive),
                        ("drift", p_drift)]:
        report(y_val_raw, preds, y_train_raw, cfg.seasonality, label=name)
    print("=" * 60)

evaluate_baselines(y_train_raw, y_val_raw, cfg)

In [ ]:
# CELL 12: Model
def get_models():
    return {
        "ridge": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("m",   Ridge(alpha=1.0)),
        ]),
        "rf": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m",   RandomForestRegressor(
                n_estimators=400, max_depth=6, min_samples_leaf=3,
                random_state=42, n_jobs=-1)),
        ]),
    }

results = {}
for name, model in get_models().items():
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    m = mase(y_val.values, pred_val, y_train_raw, cfg.seasonality)
    report(y_val, pred_val, y_train_raw, cfg.seasonality, label=f"val/{name}")
    results[name] = (m, model)

best_name = min(results, key=lambda k: results[k][0])
print(f"\n>> Model terbaik (berdasarkan VAL): {best_name} "
      f"(MASE={results[best_name][0]:.4f})")

In [ ]:
# CELL 13: Evaluasi akhir di TEST
best_model = results[best_name][1]
pred_test = best_model.predict(X_test)

report(y_test, pred_test, y_train_raw, cfg.seasonality, label=f"TEST/{best_name}")

# Diagnostik residual
res = y_test.values - pred_test
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(test[cfg.date_col], y_test.values, marker="o", label="actual")
ax[0].plot(test[cfg.date_col], pred_test, marker="x", label="pred")
ax[0].legend(); ax[0].set_title("Actual vs Pred (TEST)")
ax[1].scatter(pred_test, res, alpha=0.7); ax[1].axhline(0, color="r")
ax[1].set_title("Residual vs Pred")
ax[2].hist(res, bins=15); ax[2].set_title("Distribusi Residual")
plt.tight_layout()
plt.savefig("plots/diagnostics.png", dpi=110)
plt.show()

print(f"mean residual = {res.mean():.3f} (idealnya ~0)")
print(f"std  residual = {res.std():.3f}")

In [ ]:
# CELL 14: Walk-forward CV untuk stabilitas skor
from sklearn.model_selection import TimeSeriesSplit

def walk_forward_cv(df_feat, cfg, model):
    tscv = TimeSeriesSplit(n_splits=cfg.n_splits, gap=cfg.horizon)
    X = df_feat[X_cols].values
    y = df_feat["y_target"].values
    y_raw = df_feat[cfg.target_col].values

    scores = []
    for i, (tr, va) in enumerate(tscv.split(X)):
        m = model
        m.fit(X[tr], y[tr])
        p = m.predict(X[va])
        s = mase(y[va], p, y_raw[tr], cfg.seasonality)
        scores.append(s)
        print(f"  fold {i+1}: MASE={s:.4f}")

    print(f"\n  mean MASE = {np.mean(scores):.4f}  std = {np.std(scores):.4f}")
    return scores

walk_forward_cv(df_feat, cfg, get_models()["ridge"])

In [ ]:
# CELL 15: Checklist
checklist = {
    "Split temporal (bukan random)": True,
    "Gap = horizon di CV":           cfg.horizon >= 1,
    "Semua lag >= horizon":          all(l >= cfg.horizon for l in cfg.lags),
    "Rolling di-shift(horizon)":     True,   # lihat make_features
    "Imputer fit di train":          True,   # Pipeline menjamin
    "Scaler fit di train":           True,
    "MASE denom dari train":         True,
    "Test disentuh sekali":          True,
    "Ada baseline naive":            True,
    "Ada audit leakage fitur":       True,
}

for k, v in checklist.items():
    print(f"  [{'v' if v else 'x'}] {k}")

In [ ]:
# Cek kolom asli
print("train1 kolom :", list(train1.columns))
print("test1 kolom  :", list(test1.columns))
print("sample_submission kolom:", list(sample_submission.columns))
print()
print("train1 head:")
print(train1.head())
print()
print("test1 head:")
print(test1.head())
print()
print("sample_submission head:")
print(sample_submission.head())
print()
print("test1 dtypes:")
print(test1.dtypes)

In [ ]:
# CELL 14a: Save submission
# Sesuaikan dengan struktur sample_submission kamu
sub = pd.DataFrame({
    ID_COL: test[ID_COL].values,          # ganti ID_COL sesuai kolom id kamu
    TARGET_COL: pred_test                  # hasil prediksi
})

# Kalau sample_submission punya kolom selain id dan target, ikuti formatnya:
# sub = sample_submission.copy()
# sub[TARGET_COL] = pred_test

out_path = '/content/drive/MyDrive/Lomba Data/submissionrezala.csv'
sub.to_csv(out_path, index=False)
print(f"[save] {out_path}  shape={sub.shape}")
sub.head()

In [ ]:
# CELL 14b: Save semua hasil untuk audit
out_dir = '/content/drive/MyDrive/Lomba Data/outputs'
import os
os.makedirs(out_dir, exist_ok=True)

# 1. Prediksi test detail
test_out = test.copy()
test_out['y_pred'] = pred_test
test_out['residual'] = test_out[TARGET_COL] - test_out['y_pred']
test_out.to_csv(f'{out_dir}/test_predictions.csv', index=False)

# 2. Metrik ringkasan
from sklearn.metrics import mean_absolute_error, mean_squared_error
summary = pd.DataFrame([{
    'model': best_name,
    'MAE':  mean_absolute_error(y_test, pred_test),
    'RMSE': rmse(y_test, pred_test),
    'MASE': mase(y_test, pred_test, y_train_raw, cfg.seasonality),
    'sMAPE': smape(y_test, pred_test),
    'n_train': len(train),
    'n_val':   len(val),
    'n_test':  len(test),
}])
summary.to_csv(f'{out_dir}/metrics.csv', index=False)

# 3. Feature importance (kalau model punya)
if hasattr(best_model, 'named_steps') and 'm' in best_model.named_steps:
    inner = best_model.named_steps['m']
    if hasattr(inner, 'feature_importances_'):
        fi = pd.DataFrame({
            'feature': X_cols,
            'importance': inner.feature_importances_
        }).sort_values('importance', ascending=False)
        fi.to_csv(f'{out_dir}/feature_importance.csv', index=False)
        print(fi.head(15))

# 4. Data terpakai (untuk reproducibility)
df_feat.to_csv(f'{out_dir}/features_used.csv', index=False)

print(f"\n[save] semua output di: {out_dir}")
os.listdir(out_dir)

In [ ]:
# CELL 14c: Save model
import joblib
model_path = '/content/drive/MyDrive/Lomba Data/outputs/model.pkl'
joblib.dump({
    'model': best_model,
    'X_cols': X_cols,
    'cfg': cfg,
    'best_name': best_name,
}, model_path)
print(f"[save] model: {model_path}")

# **BATAS**

# BATAS

In [58]:
import os, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [59]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Lomba Data'
train1        = pd.read_csv(f'{BASE}/train.csv', parse_dates=['date_show'])
test1         = pd.read_csv(f'{BASE}/test.csv',  parse_dates=['date_show'])
test_history  = pd.read_csv(f'{BASE}/test_history.csv', parse_dates=['date_show'])
holidays      = pd.read_csv(f'{BASE}/holidays.csv', parse_dates=['date'])
sample_submission = pd.read_csv(f'{BASE}/sample_submission.csv')

print("train1       :", train1.shape)
print("test1        :", test1.shape)
print("test_history :", test_history.shape)
print("holidays     :", holidays.shape)
print("Train range  :", train1.date_show.min(), "->", train1.date_show.max())
print("Test range   :", test1.date_show.min(),  "->", test1.date_show.max())
print("Hist range   :", test_history.date_show.min(), "->", test_history.date_show.max())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train1       : (138959, 7)
test1        : (72611, 5)
test_history : (32323, 7)
holidays     : (366, 4)
Train range  : 2025-04-01 00:00:00 -> 2025-09-30 00:00:00
Test range   : 2025-10-04 00:00:00 -> 2026-03-27 00:00:00
Hist range   : 2025-10-01 00:00:00 -> 2026-03-20 00:00:00


In [60]:
hist = pd.concat([train1, test_history], ignore_index=True)
hist = hist.drop_duplicates(subset=['date_show','cinema_ids','movie_title'], keep='last')
hist = hist.sort_values('date_show').reset_index(drop=True)
print("History:", hist.shape, "|", hist.date_show.min(), "->", hist.date_show.max())

History: (171282, 7) | 2025-04-01 00:00:00 -> 2026-03-20 00:00:00


In [61]:
holidays_clean = holidays[['date','day_tipe','holiday_tipe']].rename(columns={'date':'date_show'})
holidays_clean['date_show'] = pd.to_datetime(holidays_clean['date_show'])
holidays_clean = holidays_clean.drop_duplicates('date_show')

# STATISTIK dari TRAIN saja (bukan dari test_history — biar tidak "tahu masa depan")
movie_stats  = train1.groupby('movie_title')['total_ticket'].agg(
    movie_mean='mean', movie_std='std', movie_cnt='count').reset_index()
cinema_stats = train1.groupby('cinema_ids')['total_ticket'].agg(
    cinema_mean='mean', cinema_std='std').reset_index()
pair_stats   = train1.groupby(['cinema_ids','movie_title'])['total_ticket'].agg(
    pair_mean='mean', pair_cnt='count').reset_index()
city_stats   = train1.groupby('city_name')['total_ticket'].agg(city_mean='mean').reset_index()

def build_features(base_df, hist, holidays_clean,
                   movie_stats, cinema_stats, pair_stats, city_stats):
    df = base_df.copy()
    df['date_show'] = pd.to_datetime(df['date_show'])
    df = df.sort_values('date_show').reset_index(drop=True)

    # Kalender
    df['dow']        = df['date_show'].dt.dayofweek
    df['month']      = df['date_show'].dt.month
    df['day']        = df['date_show'].dt.day
    df['weekofyear'] = df['date_show'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

    # Holiday
    df = df.merge(holidays_clean, on='date_show', how='left')
    df['is_holiday']   = (df['holiday_tipe'] == 'holiday').astype(int)
    df['day_tipe']     = df['day_tipe'].fillna('normal')
    df['holiday_tipe'] = df['holiday_tipe'].fillna('normal')

    # Group stats
    df = df.merge(movie_stats,  on='movie_title', how='left')
    df = df.merge(cinema_stats, on='cinema_ids',  how='left')
    df = df.merge(pair_stats,   on=['cinema_ids','movie_title'], how='left')
    df = df.merge(city_stats,   on='city_name',   how='left')

    # Lag per (cinema, movie) pakai merge_asof — hanya data SEBELUM tanggal
    hl = hist[['date_show','cinema_ids','movie_title','total_ticket']].copy()
    hl = hl.sort_values('date_show').reset_index(drop=True)

    df = df.sort_values('date_show').reset_index(drop=True)

    # last known ticket untuk pair ini
    df = pd.merge_asof(
        df,
        hl.rename(columns={'total_ticket':'pair_last'}),
        on='date_show', by=['cinema_ids','movie_title'],
        direction='backward', allow_exact_matches=False
    )

    # 7 hari yang lalu
    hl7 = hl.copy(); hl7['date_show'] = hl7['date_show'] + pd.Timedelta(days=7)
    df = pd.merge_asof(
        df,
        hl7.rename(columns={'total_ticket':'pair_7d_ago'}).sort_values('date_show'),
        on='date_show', by=['cinema_ids','movie_title'],
        direction='backward', allow_exact_matches=False
    )

    # 28 hari yang lalu
    hl28 = hl.copy(); hl28['date_show'] = hl28['date_show'] + pd.Timedelta(days=28)
    df = pd.merge_asof(
        df,
        hl28.rename(columns={'total_ticket':'pair_28d_ago'}).sort_values('date_show'),
        on='date_show', by=['cinema_ids','movie_title'],
        direction='backward', allow_exact_matches=False
    )
    return df

In [62]:
train_feat = build_features(train1, hist, holidays_clean,
                            movie_stats, cinema_stats, pair_stats, city_stats)
test_feat  = build_features(test1,  hist, holidays_clean,
                            movie_stats, cinema_stats, pair_stats, city_stats)

print("train_feat:", train_feat.shape)
print("test_feat :", test_feat.shape)

train_feat: (138959, 26)
test_feat : (72611, 24)


In [63]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(pd.concat([train_feat['day_tipe'], test_feat['day_tipe']]))
train_feat['day_tipe_enc'] = le.transform(train_feat['day_tipe'])
test_feat['day_tipe_enc']  = le.transform(test_feat['day_tipe'])

FEATURE_COLS = ['dow','month','day','weekofyear','is_weekend','is_holiday',
                'movie_mean','movie_std','movie_cnt',
                'cinema_mean','cinema_std',
                'pair_mean','pair_cnt','city_mean',
                'pair_last','pair_7d_ago','pair_28d_ago',
                'day_tipe_enc']

X_train = train_feat[FEATURE_COLS]
y_train = train_feat['total_ticket'].values
X_test  = test_feat[FEATURE_COLS]
print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (138959, 18) X_test: (72611, 18)


In [ ]:
cutoff = train_feat['date_show'].max() - pd.Timedelta(days=30)
tr_mask = (train_feat['date_show'] <= cutoff).values
va_mask = (train_feat['date_show'] >  cutoff).values

imputer = SimpleImputer(strategy='median')
Xtr = imputer.fit_transform(X_train.values[tr_mask])
Xva = imputer.transform(X_train.values[va_mask])

model = RandomForestRegressor(
    n_estimators=300, max_depth=12, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)
model.fit(Xtr, y_train[tr_mask])
pred_va = model.predict(Xva)

print(f"Val MAE : {mean_absolute_error(y_train[va_mask], pred_va):.3f}")
print(f"Val RMSE: {np.sqrt(mean_squared_error(y_train[va_mask], pred_va)):.3f}")

In [ ]:
Xfull = imputer.fit_transform(X_train.values)
model.fit(Xfull, y_train)
print("Model retrained.")

In [ ]:
Xte = imputer.transform(X_test.values)
pred_test = model.predict(Xte)
pred_test = np.clip(pred_test, 0, None)

sub = pd.DataFrame({'id': test1['id'].values, 'total_ticket': pred_test})
out_path = f'{BASE}/submissionrezala.csv'
sub.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(sub.head())
print(sub.describe())